In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from scipy.stats import loguniform


# Data loading and cleaning

In [12]:
diamonds = pd.read_csv("../data/diamonds.csv")
# diamonds.info()
X = diamonds.drop("price", axis=1)
y = diamonds["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clarity_order = [
    "I3", "I2", "I1",
    "SI2", "SI1",
    "VS2", "VS1",
    "VVS2", "VVS1",
    "IF", "FL"
]

cut_order = [
    "Fair", "Good",
    "Very Good", "Premium",
    "Ideal"
]

color_order = [
    "J", "I", "H",
    "G", "F", "E", "D"
]

num_pipeline = make_pipeline(
    StandardScaler()
)

ordinal_pipeline = make_pipeline(
    OrdinalEncoder(categories=[clarity_order, cut_order, color_order]),
    StandardScaler()
)


preprocessing = ColumnTransformer([
    ("num", num_pipeline, list(X_train.select_dtypes(include=["int64", "float64"]).columns)),
    ("clarity", ordinal_pipeline, ["clarity", "cut", "color"])
])

# Model selection with Randomized Seatch for alpha

In [13]:
reg = make_pipeline(
    preprocessing,
    Ridge()
)

param_distribs = {
    "ridge__alpha": loguniform(1e-4, 1e4)
}

rndsearch = RandomizedSearchCV(
    reg,
    param_distributions=param_distribs,
    n_iter=30,
    scoring="r2",
    cv=5,
    random_state=42,
    n_jobs=-1
)

rndsearch.fit(X_train, y_train)

print("Best params:", rndsearch.best_params_)
print(f"Best CV R2: {rndsearch.best_score_:.4f}")

Best params: {'ridge__alpha': np.float64(7.849159562555091)}
Best CV R2: 0.9074


# Prediction

In [14]:
best_model = rndsearch.best_estimator_

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)


print("Test R2:", r2_score(y_test, y_pred))
print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test RMSE:", mean_squared_error(y_test, y_pred) ** 0.5)
print("Test MAE:", mean_absolute_error(y_test, y_pred))

Test R2: 0.906107687668137
Test MSE: 1492589.3247855278
Test RMSE: 1221.7157299411053
Test MAE: 805.5263035672473
